# core

> Usage counting, the callback protocol, and how a reply renders.

Nothing here knows about a model. These are the pieces every backend hands back and every caller reads, so they are defined once, before anything that produces them.

In [ ]:
#| default_exp core

In [ ]:
#| export
import json, os
from html import escape
from fastcore.all import store_attr, GetAttr, L

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail

## System certificates

A TLS-intercepting proxy breaks every hosted backend at once, and the first sign of it is an unrelated-looking handshake error deep inside an HTTP client. Verifying against the OS trust store instead of `certifi` fixes that, so urai does it on import rather than waiting to be asked.

In [ ]:
#| export
def use_system_certs(force=False):
    "Verify TLS against the OS trust store rather than `certifi`. False if `truststore` is missing, or off by `URAI_SYSTEM_CERTS=0`."
    off = os.getenv('URAI_SYSTEM_CERTS', os.getenv('RISHI_SYSTEM_CERTS', '1')) == '0'
    if not force and off: return False
    try: import truststore
    except ImportError: return False
    truststore.inject_into_ssl()    # idempotent: it rebinds `ssl.SSLContext` to its own
    return True

system_certs = use_system_certs()   #: done once, on import

In [ ]:
os.environ['URAI_SYSTEM_CERTS'] = '0'
test_eq(use_system_certs(), False)          # opted out
test_eq(use_system_certs(force=True), True) # ...and `force` overrides the opt-out
del os.environ['URAI_SYSTEM_CERTS']

## Usage

One turn's token counts. Local inference reports prompt and completion tokens and nothing else; hosted providers add cached reads, cache writes, reasoning tokens and a price. `UsageStats` carries all of them so a caller mixing local and hosted models handles one type rather than two, and the fields a backend cannot fill stay at zero.

Adding is how a turn's usage folds into a session's, so `sum` over a list of turns works and `None` is absorbed.

In [ ]:
#| export
class UsageStats:
    "Token usage for a chat turn. `cost` and `model` are always present, so local and hosted usage share one type."
    _sums = ('prompt_tokens', 'completion_tokens', 'total_tokens', 'n', 'cached_tokens', 'cost',
             'reasoning_tokens', 'cache_creation_tokens')
    def __init__(self,
                 prompt_tokens=0,           # tokens in the prompt
                 completion_tokens=0,       # tokens the model generated
                 total_tokens=0,            # prompt + completion, as the backend reports it
                 n=0,                       # turns folded into this total
                 cached_tokens=0,           # prompt tokens served from a KV or prefix cache
                 cost=0.0,                  # what the provider charged; 0 for local inference
                 model=None,                # which model spent them
                 reasoning_tokens=0,        # billed separately by hosted providers
                 cache_creation_tokens=0):  # ...as are cache writes
        store_attr()

    def __add__(self, other):
        if other is None: return self
        return UsageStats(**{k: getattr(self, k) + getattr(other, k) for k in self._sums},
                          model=self.model or other.model)
    def __radd__(self, other): return self if other in (None, 0) else self.__add__(other)

    def __repr__(self):
        p = [f'total={self.total_tokens:,}', f'in={self.prompt_tokens:,}',
             f'out={self.completion_tokens:,}', f'turns={self.n}']
        for k, lbl in (('cached_tokens','cached'), ('reasoning_tokens','reasoning'),
                       ('cache_creation_tokens','cache_write')):
            if (v := getattr(self, k)): p.append(f'{lbl}={v:,}')
        if self.cost: p.append(f'cost=${self.cost:,.4f}')
        if self.model: p.append(f'model={self.model}')
        return '|'.join(p)

    def fmt(self):
        "Markdown `<details>` token block, empty when nothing was spent."
        if not self.total_tokens: return ''
        return f"\n\n<details><summary>{self.total_tokens:,} tokens</summary>\n\n`{self!r}`\n\n</details>\n"

In [ ]:
a = UsageStats(prompt_tokens=10, completion_tokens=5, total_tokens=15, n=1, model='gpt-5.1')
b = UsageStats(prompt_tokens=3, completion_tokens=2, total_tokens=5, n=1)
tot = a + b
test_eq((tot.prompt_tokens, tot.completion_tokens, tot.total_tokens, tot.n), (13, 7, 20, 2))
test_eq(tot.model, 'gpt-5.1')       # the first model named wins; `b` did not say
test_eq(sum([a, b]).total_tokens, 20)   # `sum` starts at 0, which `__radd__` absorbs
test_eq(a + None, a)

In [ ]:
test_eq(repr(b), 'total=5|in=3|out=2|turns=1')
test_eq(repr(UsageStats(total_tokens=9, cost=0.25, cached_tokens=4)),
        'total=9|in=0|out=0|turns=0|cached=4|cost=$0.2500')
test_eq(UsageStats().fmt(), '')      # nothing spent, nothing shown

## Callbacks

A callback reads chat state as its own attributes: `GetAttr` forwards anything it does not define to `self.chat`, so `self.turn_res` inside a callback means `chat.turn_res`. That keeps the bodies short and means a callback never has to be handed the state it wants.

`run_cbs` dispatches one event name. Callbacks fire in `order`, a callback with `run=False` is skipped, and anything a callback yields is forwarded to the caller — which is how a callback injects text into a stream.

In [ ]:
#| export
class ChatCallback(GetAttr):
    "Base chat callback. Reads chat state via `GetAttr`, so `self.turn_msg` is `chat.turn_msg`."
    order, _default, chat, run = 0, 'chat', None, True
    def __repr__(self): return type(self).__name__

def run_cbs(chat, event):
    "Dispatch `event` to enabled callbacks in `order`, forwarding whatever they yield."
    for cb in chat.cbs.sorted('order'):
        if cb.run and hasattr(cb, event):
            r = getattr(cb, event)()
            if r is not None: yield from r

In [ ]:
class _Chat: pass
class _Loud(ChatCallback):
    order = 1
    def after_response(self): yield f'loud saw {self.turn_res}'
class _Quiet(ChatCallback):
    order = 0
    def after_response(self): yield 'quiet first'

c = _Chat(); c.turn_res = 'hi'; c.cbs = L(_Loud(), _Quiet())
for cb in c.cbs: cb.chat = c
test_eq(list(run_cbs(c, 'after_response')), ['quiet first', 'loud saw hi'])
test_eq(list(run_cbs(c, 'before_send')), [])   # no callback defines it

In [ ]:
c.cbs[0].run = False
test_eq(list(run_cbs(c, 'after_response')), ['quiet first'])
test_eq(repr(_Quiet()), '_Quiet')

## Reading a reply

A reply is a plain dict. Its `content` is either a string or a list of typed parts, and `resp_text` joins the text ones so a caller never has to branch on which shape arrived.

In [ ]:
#| export
def resp_text(resp):
    "The text of a response or chunk dict, whichever content shape it uses."
    c = resp.get('content', []) if isinstance(resp, dict) else ''
    if isinstance(c, str): return c
    return ''.join(p.get('text', '') for p in c if isinstance(p, dict) and p.get('type') == 'text')

def thought(resp):
    "The model's thinking (`channels.thought`), or ''."
    return resp.get('channels', {}).get('thought', '') if isinstance(resp, dict) else ''

def truncated(resp):
    "Was `resp` flagged as cut off at the token cap?"
    return bool(resp.get('truncated')) if isinstance(resp, dict) else False

def has_tool_call(o):
    "Does this chunk carry a `tool_call` content part?"
    return any(isinstance(p, dict) and p.get('type') == 'tool_call'
               for p in (o.get('content') or [] if isinstance(o, dict) else []))

In [ ]:
test_eq(resp_text({'content': 'plain'}), 'plain')
test_eq(resp_text({'content': [{'type':'text','text':'a'}, {'type':'tool_call','name':'f'},
                               {'type':'text','text':'b'}]}), 'ab')
test_eq(resp_text(None), '')
test_eq(thought({'channels': {'thought': 'hmm'}}), 'hmm')
test_eq(thought({}), '')
test_eq(truncated({'truncated': True}), True)
test_eq(has_tool_call({'content': [{'type':'tool_call','name':'f'}]}), True)
test_eq(has_tool_call({'content': 'text only'}), False)

## Rendering

`Resp` is that same dict with a notebook renderer attached, so a returned reply displays as markdown instead of as a wall of JSON. Thinking becomes a blockquote, because every markdown engine renders one and no engine renders a custom tag.

In [ ]:
#| export
def quote_(text):
    "`text` as a markdown blockquote under a Thinking header."
    return '> **🧠 Thinking**\n>\n' + '\n'.join('> ' + l for l in text.splitlines())

def tc_summary_(name, args, result=None):
    "One-line `<code>` summary of a tool call."
    params = ', '.join(f'{k}={v!r}' for k, v in (args or {}).items())
    res = f' -> {result}' if result is not None else ''
    return '<code>' + escape(f'{name}({params}){res}') + '</code>'

def mk_tr_details(name, args, result, mx=2000):
    "`<details>` JSON block for a completed tool call."
    body = json.dumps({'call': {'function': name, 'arguments': args}, 'result': str(result)[:mx]}, indent=2)
    return (f'\n\n<details><summary>{tc_summary_(name, args, result)}</summary>\n\n'
            f'```json\n{body}\n```\n\n</details>\n\n')

class Resp(dict):
    "A response dict that renders as markdown in a notebook."
    def _repr_markdown_(self):
        md = ''
        if th := thought(self): md += quote_(th) + '\n\n'
        md += resp_text(self)
        for tc in self.get('tool_calls', []):
            fn = tc.get('function', {})
            md += f"\n\n🔧 {fn.get('name','')}({fn.get('arguments', {})})"
        for p in (self.get('content') or []):
            if isinstance(p, dict) and p.get('type') == 'tool_response':
                md += f"\n\n↩︎ **{p.get('name','')}**: {p.get('response')}"
        return md or '*(no text)*'

In [ ]:
test_eq(quote_('one\ntwo'), '> **🧠 Thinking**\n>\n> one\n> two')
test_eq(tc_summary_('add', {'a': 1}), '<code>add(a=1)</code>')
test_eq(tc_summary_('esc', {'s': '<b>'}), '<code>esc(s=&#x27;&lt;b&gt;&#x27;)</code>')  # escaped for html
assert '```json' in mk_tr_details('add', {'a': 1}, 3)

In [ ]:
test_eq(Resp({'content': 'hi'})._repr_markdown_(), 'hi')
test_eq(Resp({})._repr_markdown_(), '*(no text)*')
r = Resp({'content': 'done', 'channels': {'thought': 'why'},
          'tool_calls': [{'function': {'name': 'add', 'arguments': {'a': 1}}}]})
md = r._repr_markdown_()
assert md.startswith('> **🧠 Thinking**') and 'done' in md and '🔧 add' in md

## Streaming

`StreamFormatter` turns a stream of chunk dicts into a stream of markdown strings. It is stateful for one reason: a blockquote has to stay open across every thinking chunk and close on the first text chunk, and a chunk on its own cannot know which of those it is.

In [ ]:
#| export
class StreamFormatter:
    "Format a chunk stream to markdown. Thinking streams as one blockquote."
    def __init__(self, mx=2000, showthink=True, showcalls=False):
        self.outp = ''; self._inthink = False; store_attr()

    def format_item(self, o):
        "Format one chunk dict: thinking, text, or a tool call."
        res = ''
        if (th := thought(o)) and self.showthink:
            if not self._inthink: res += '> **🧠 Thinking**\n>\n> '; self._inthink = True
            res += th.replace('\n', '\n> ')
        if txt := resp_text(o):
            if self._inthink: res += '\n\n'; self._inthink = False
            res += txt
        if self.showcalls:
            for p in (o.get('content', []) if isinstance(o, dict) else []):
                if isinstance(p, dict) and p.get('type') == 'tool_call':
                    res += f"\n- ⏳ {tc_summary_(p.get('name', ''), p.get('arguments', {}))}\n"
        self.outp += res
        return res

    def format_stream(self, rs):
        "Yield markdown for each chunk, closing any open thinking blockquote at the end."
        for o in rs: yield self.format_item(o)
        if self._inthink: self._inthink = False; yield '\n\n'

def display_stream(chunks):
    "Render a markdown-chunk stream live in a notebook, and return the full markdown."
    from IPython.display import display, Markdown
    h, md = display(Markdown(''), display_id=True), ''
    for c in chunks:
        md += c
        if h is not None: h.update(Markdown(md))
    return md

In [ ]:
chunks = [{'channels': {'thought': 'let me'}}, {'channels': {'thought': ' see'}},
          {'content': 'the answer'}]
test_eq(''.join(StreamFormatter().format_stream(chunks)),
        '> **🧠 Thinking**\n>\n> let me see\n\nthe answer')

In [ ]:
f = StreamFormatter()
test_eq(''.join(f.format_stream([{'channels': {'thought': 'just thinking'}}])),
        '> **🧠 Thinking**\n>\n> just thinking\n\n')   # closed at the end of the stream
test_eq(f.outp, '> **🧠 Thinking**\n>\n> just thinking')  # ...but the close is not part of the text
test_eq(''.join(StreamFormatter(showthink=False).format_stream(chunks)), 'the answer')

In [ ]:
call = [{'content': [{'type': 'tool_call', 'name': 'add', 'arguments': {'a': 1}}]}]
test_eq(''.join(StreamFormatter(showcalls=True).format_stream(call)),
        '\n- ⏳ <code>add(a=1)</code>\n')
test_eq(''.join(StreamFormatter().format_stream(call)), '')   # calls are hidden by default

## Stock callbacks

Two that need nothing but what is above. The rest live with the loop that fires them.

In [ ]:
#| export
tool_reminder_ = ('\n<system-reminder>After every tool call result, briefly summarise in prose '
                  'what you found before continuing or calling another tool.</system-reminder>')

class TruncationCallback(ChatCallback):
    "Flag `turn_res['truncated']` when a reply reaches the output cap. Best effort: some backends do not say."
    order = 20
    def __init__(self, max_tokens): store_attr()
    def after_response(self):
        if self.chat.use.completion_tokens >= self.max_tokens: self.chat.turn_res['truncated'] = True

In [ ]:
c = _Chat(); c.use = UsageStats(completion_tokens=100); c.turn_res = Resp({'content': 'cut off'})
c.cbs = L(TruncationCallback(max_tokens=100)); c.cbs[0].chat = c
list(run_cbs(c, 'after_response'))
test_eq(truncated(c.turn_res), True)

In [ ]:
c.use = UsageStats(completion_tokens=99); c.turn_res = Resp({'content': 'room to spare'})
list(run_cbs(c, 'after_response'))
test_eq(truncated(c.turn_res), False)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()